In [32]:

# This code is to check duplicates in a newly imported Transactions tab in my Expenses.xls against any expenses 
# already recorded in a holiday tab in R2T Reconciliation.

# The flow is:
# 1. Read in the Transactions tab from Expenses.xls and the holiday tab from R2T Reconciliation.
# 2. For each transaction in the Transactions tab, check if there is a matching transaction in the holiday tab.
# 3. If a match is found, print out the details of the duplicate transaction.



In [33]:
# import libraries
import pandas as pd
import numpy as np
import thefuzz
from thefuzz import process


In [34]:
# import data from Excel
excel_file = r"C:\Users\tombe\Google Drive\Expenses_New.xlsm"
df_data = pd.read_excel(excel_file, sheet_name="Transactions", header=0)
df_dups = pd.read_excel(excel_file, sheet_name="Possible Duplicates", header=0)

# clean column names in df_data
df_data.columns = (
    df_data.columns
      .str.strip()        # remove leading/trailing spaces
      .str.lower()        # lowercase
      .str.replace(' ', '_')  # replace spaces with underscores
)

# clean column names in df_dups
df_dups.columns = (
    df_dups.columns
      .str.replace(' ', '_')  # replace spaces with underscores
      .str.strip()        # remove leading/trailing spaces
      .str.lower()        # lowercase
      
)

In [35]:
# convert transaction_date to datetime
df_data['transaction_date'] = pd.to_datetime(df_data['transaction_date'])

# remove partial blank rows from df_dups
df_dups = df_dups [~df_dups['cost_in_origin_currency'].isna()]

In [36]:
# convert date in df_dups to string
df_dups['date'] = df_dups['date'].astype(str)

# create a mask to identify rows where 'date' contains 'Basics'
mask = df_dups['date'].str.contains('Basics', na=False)

# extract day from 'date' column, convert to float, and handle non-numeric values by coercing them to NaN (using errors='coerce')
# to avoid errors during the conversion and ensure that non-numeric values are handled gracefully without causing the program to crash. 
day = df_dups['date'].str.extract(r'(\d+)', expand=False).astype(float)
# In this line, if non-numeric values are present in the 'date' column, they will be converted to NaN, 
# and the subsequent operations will be performed on valid numeric values only.

month = np.where((day >= 8) & (day <= 31), 12, 1)
year = np.where(month == 12, 2025, 2026)

df_dups['clean_date'] = pd.to_datetime(
    dict(year=year, month=month, day=day), # the dict() function is used to create a dictionary with keys 'year', 'month', and 'day' and corresponding values from the year, month, and day arrays. This dictionary is then passed to the pd.to_datetime() function to create a datetime object.
    errors='coerce'
)

# set 'clean_date' to NaT for rows where 'date' contains 'Basics'
df_dups.loc[mask, 'clean_date'] = pd.NaT


In [ ]:
# now we do the matching of transactions in df_data against transactions in df_dups using thefuzz library.
# We will use the process.extract() function to find the best matches for each transaction in df_data based on the 
# 'description' column. We will also set a threshold for the matching score to filter out low-quality matches.
import recordlinkage

indexer = recordlinkage.Index()

indexer.block(left_on = 'transaction_date', right_on = 'clean_date')

pairs = indexer.index(df_data, df_dups)

compare_cl = recordlinkage.Compare()
compare_cl.numeric("billing_amount", "cost_in_origin_currency", label="cost",method="gauss", offset=0.01, scale=0.02)
compare_cl.string("merchant", "detail", method="jarowinkler", label="description")

initial_matches = compare_cl.compute(pairs, df_data, df_dups)

initial_matches["score"] = initial_matches.sum(axis=1)
initial_matches = initial_matches [initial_matches["score"] >= 0.4]

final_matches_by_tuple = initial_matches.groupby(level=0)["score"].idxmax()

# Here we filtering initial_matches by the final matches (to keep the score), and by using .reset_index() to 
# convert the indices back to columns for easier readability and analysis.
df_final_matches = initial_matches.loc[final_matches_by_tuple].reset_index()
# Now re-name columns in df_final_matches to be more descriptive and easier to understand.
df_final_matches.rename(columns={"level_0": "left_idx", "level_1": "right_idx"}, inplace=True)
df_final_matches.set_index('left_idx', inplace=True)

In [38]:
print (final_matches_by_tuple)
print (df_data['billing_amount'].dtype)
print (df_dups['cost_in_origin_currency'].dtype)

65     (65, 108)
66     (66, 110)
67     (67, 108)
68     (68, 109)
69     (69, 108)
         ...    
132     (132, 5)
133     (133, 6)
134     (134, 7)
135     (135, 8)
136     (136, 8)
Name: score, Length: 63, dtype: object
float64
float64


In [ ]:
df_merged = df_data.join(df_final_matches, how='left')

df_merged = df_merged.merge(df_dups, how='left', left_on='right_idx', right_index=True, suffixes=('_data', '_dups'))

In [43]:
print (df_merged.tail(20))

     transaction_date posting_date  billing_amount                merchant  \
44.0       2025-12-16   2025-12-17            7.50               The Habit   
45.0       2025-12-16   2025-12-17           12.00  ARRIVA YORKSHIRE LIMIT   
46.0       2025-12-16   2025-12-17            1.00              UBER *TRIP   
47.0       2025-12-16   2025-12-17            7.15        CROSS KEYS HOTEL   
48.0       2025-12-16   2025-12-17            5.55           PRET A MANGER   
49.0       2025-12-16   2025-12-17            1.85           PRET A MANGER   
50.0       2025-12-16   2025-12-17           15.00     YORK PAVILION HOTEL   
51.0       2025-12-15   2025-12-16          110.47  TRAVELODG TRAVELODGE G   
52.0       2025-12-15   2025-12-16           18.75       TESCO STORES 2171   
53.0       2025-12-15   2025-12-16            5.44              BOOTS 1667   
54.0       2025-12-15   2025-12-16           18.19    SUPERDRUG STORES PLC   
55.0       2025-12-15   2025-12-16            9.10  CAFFE NERO B

In [40]:
# with pd.ExcelWriter(
#    excel_file,
#    engine="openpyxl",
#    mode="a",
#    if_sheet_exists="replace"
#) as writer:
    
#    df_merged.to_excel(
#        writer,
#        sheet_name="Merged_Data",
#        index=True   # keep index if useful
#    )

df_merged.to_excel(
    r"C:\Users\tombe\Google Drive\Expenses_Merged_Test.xlsx",
    index=False
)   

In [41]:
# create new df re-arranged

# output_path= r"C:\Users\tombe\Downloads\cc_statement_rearranged.xlsx"

# cc_df['description'] = cc_df['merchant'] + " - " + cc_df['merchant_city']

# T_col = ["T"] * len(cc_df)  # one "T" per row
# Y_col = ["Y"] * len(cc_df)  # one "Y" per row


# with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
#     cc_df[['transaction_date']].to_excel(
#         writer,
#         sheet_name="Sheet1",
#         startcol=0,   # Column A
#         index=False
#     )
#     cc_df[['billing_amount']].to_excel(
#         writer,
#         sheet_name="Sheet1",
#         startcol=1,   # Column B
#         index=False
#     )
#     pd.DataFrame({"T_col": T_col}).to_excel(
#         writer,
#         sheet_name="Sheet1",
#         startcol=4,  # Column E
#         index=False
#     )
#     pd.DataFrame({"Y_col": Y_col}).to_excel(
#         writer,
#         sheet_name="Sheet1",
#         startcol=5,  # Column F
#         index=False
#     )
#     cc_df[['description']].to_excel(
#         writer,
#         sheet_name="Sheet1",
#         startcol=6,   # Column G
#         index=False
#     )





In [3]:
len?

Signature: len(obj, /)
Docstring: Return the number of items in a container.
Type:      builtin_function_or_method

In [11]:

print(inspect.getsource(len))


TypeError: module, class, method, function, traceback, frame, or code object was expected, got builtin_function_or_method